## Rule-Based Fraud Detection Example

This example demonstrates a basic rule-based fraud detection system using a synthetic dataset. We'll create some transaction data and then define simple rules to flag potentially fraudulent transactions.

In [1]:
import pandas as pd
import numpy as np

# 1. Create a synthetic dataset
np.random.seed(42)
num_transactions = 100

data = {
    'transaction_id': range(1, num_transactions + 1),
    'user_id': np.random.randint(1, 20, num_transactions),
    'amount': np.random.normal(loc=50, scale=30, size=num_transactions).round(2).clip(min=1),
    'location': np.random.choice(['US', 'CA', 'GB', 'DE', 'JP'], num_transactions, p=[0.7, 0.1, 0.1, 0.05, 0.05]),
    'time_of_day_hour': np.random.randint(0, 24, num_transactions),
    'is_international': np.random.choice([True, False], num_transactions, p=[0.15, 0.85]),
    'transaction_type': np.random.choice(['online', 'in-store'], num_transactions, p=[0.6, 0.4])
}

df = pd.DataFrame(data)

# Introduce some 'anomalies' for demonstration
df.loc[0, 'amount'] = 1500  # High amount
df.loc[1, 'is_international'] = True # International
df.loc[2, 'time_of_day_hour'] = 3 # Late night
df.loc[3, ['amount', 'is_international']] = [800, True] # High amount & international

print("Original Dataset Head:")
display(df.head())


Original Dataset Head:


,transaction_id,user_id,amount,location,time_of_day_hour,is_international,transaction_type
0,1,7,1500.00,US,17,False,online
1,2,15,1.00,DE,9,True,in-store
2,3,11,59.81,JP,3,False,in-store
3,4,8,800.00,US,16,True,online
4,5,7,64.03,US,23,False,in-store


### Define and Apply Fraud Detection Rules

We'll set up some simple rules to identify potentially fraudulent transactions:

1.  **High Value Transaction**: Flag transactions over a certain amount (e.g., $500).
2.  **Unusual Time**: Flag transactions occurring between 1 AM and 5 AM.
3.  **International Transaction**: Flag international transactions (these might warrant extra scrutiny, especially combined with other factors).
4.  **Combination Rule**: Flag transactions that are both high value and international.

In [2]:
# 2. Define and apply rules for fraud detection

def detect_fraud(row):
    is_fraud = False
    reason = []

    # Rule 1: High Value Transaction
    if row['amount'] > 500:
        is_fraud = True
        reason.append("High Value Transaction")

    # Rule 2: Unusual Time of Day (e.g., 1 AM to 5 AM)
    if 1 <= row['time_of_day_hour'] <= 5:
        is_fraud = True
        reason.append("Unusual Time of Day")

    # Rule 3: International Transaction
    if row['is_international']:
        is_fraud = True
        reason.append("International Transaction")

    # You can add more complex rules or combinations
    if row['amount'] > 200 and row['is_international']:
        is_fraud = True
        reason.append("High Value International Transaction")

    return is_fraud, "; ".join(reason)

# Apply the detection function to each row
df[['is_fraud', 'fraud_reason']] = df.apply(lambda row: pd.Series(detect_fraud(row)), axis=1)

print("Dataset with Fraud Flags:")
display(df.head(10))


Dataset with Fraud Flags:


,transaction_id,user_id,amount,location,time_of_day_hour,is_international,transaction_type,is_fraud,fraud_reason
0,1,7,1500.00,US,17,False,online,True,High Value Transaction
1,2,15,1.00,DE,9,True,in-store,True,International Transaction
2,3,11,59.81,JP,3,False,in-store,True,Unusual Time of Day
3,4,8,800.00,US,16,True,online,True,High Value Transaction; International Transact...
4,5,7,64.03,US,23,False,in-store,False,
5,6,19,72.08,US,18,False,online,False,
6,7,11,26.61,US,22,False,in-store,False,
7,8,11,24.68,JP,4,False,online,True,Unusual Time of Day
8,9,4,45.48,US,20,False,online,False,
9,10,8,21.03,US,22,False,in-store,False,


### Summary of Fraudulent Transactions

Let's see how many transactions were flagged and why.

In [3]:
# 3. Display results

fraudulent_transactions = df[df['is_fraud']]

print(f"Total transactions: {len(df)}")
print(f"Flagged fraudulent transactions: {len(fraudulent_transactions)}")

if not fraudulent_transactions.empty:
    print("\nDetails of Flagged Transactions:")
    display(fraudulent_transactions)
else:
    print("\nNo transactions were flagged as potentially fraudulent based on the defined rules.")

print("\nDistribution of Fraud Reasons:")
display(fraudulent_transactions['fraud_reason'].value_counts())


Total transactions: 100
Flagged fraudulent transactions: 41

Details of Flagged Transactions:


,transaction_id,user_id,amount,location,time_of_day_hour,is_international,transaction_type,is_fraud,fraud_reason
0,1,7,1500.00,US,17,False,online,True,High Value Transaction
1,2,15,1.00,DE,9,True,in-store,True,International Transaction
2,3,11,59.81,JP,3,False,in-store,True,Unusual Time of Day
3,4,8,800.00,US,16,True,online,True,High Value Transaction; International Transact...
7,8,11,24.68,JP,4,False,online,True,Unusual Time of Day
16,17,12,76.18,US,1,False,online,True,Unusual Time of Day
21,22,15,76.90,JP,20,True,in-store,True,International Transaction
25,26,3,43.13,US,4,False,in-store,True,Unusual Time of Day
28,29,7,26.78,GB,2,True,in-store,True,Unusual Time of Day; International Transaction
30,31,7,71.72,US,15,True,in-store,True,International Transaction



Distribution of Fraud Reasons:


,count
fraud_reason,
International Transaction,17
Unusual Time of Day,17
Unusual Time of Day; International Transaction,5
High Value Transaction,1
High Value Transaction; International Transaction; High Value International Transaction,1
